# Behavior Explainer — Results Notebook

This notebook reproduces the metrics and figures from the paper.
No model training or explanation enumeration is required — all results are precomputed
and stored as CSV files in `intermediate_results/`.

**How to use:** edit the *Configuration* cell (shared across all metrics), then run all cells.

---

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
from IPython.display import display

# Make the analysis package importable from the repo root
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

INTERMEDIATE_RESULTS = REPO_ROOT / 'intermediate_results'
RESULTS_DIR          = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## Configuration

Edit the variables below — they are shared across all metrics in this notebook.

In [ ]:
# ── Experiment settings ───────────────────────────────────────────────────────
model      = 'resnet'    # vision model  (e.g. 'resnet', 'vgg')
dataset    = 'rival10'   # dataset       (e.g. 'rival10', 'eurosat')
vocab_size = 300         # vocabulary size (e.g. 200, 300)

# ── Behavior type ─────────────────────────────────────────────────────────────
# 'correct'           → correctly classified images of class_a   (B2<class_a>)
# 'misclassification' → images of class_a misclassified as class_b (B5<class_a><class_b>)
behavior_type = 'correct'
class_a = 6    # target class index
class_b = None # only needed when behavior_type == 'misclassification'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Build behavior identifier
if behavior_type == 'correct':
    behavior = f'B2{class_a}'
elif behavior_type == 'misclassification':
    if class_b is None:
        raise ValueError('Set class_b for misclassification behavior')
    behavior = f'B5{class_a}{class_b}'
else:
    raise ValueError(f"behavior_type must be 'correct' or 'misclassification'")

print(f'Behavior: {behavior}  |  Model: {model}  |  Dataset: {dataset}  |  N={vocab_size}')

---
## Metric 1 — Average Number of Explanations per Image

Reports the mean ± standard deviation of:
- **AXps** — abductive explanations (*why* the model predicts what it predicts)
- **CXps** — contrastive explanations (*why not* an alternative class)

across **9 configurations**: 3 erasure algorithms (Ortho, SpLiCE, LEACE) ×
3 enumeration algorithms (XpEnum, XpSatEnum, NaiveEnum).

> `*` = partial run (timeout reached before all images were processed)   `-` = not available

In [ ]:
from analysis.avg_xp_count import compute_avg_xp_count

df_m1 = compute_avg_xp_count(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)

caption = f'Avg ± std explanations / image — behavior {behavior} ({model}, {dataset}, N={vocab_size})'
df_m1.style.set_caption(caption)

In [ ]:
out = RESULTS_DIR / f'metric1_{behavior}.csv'
df_m1.to_csv(out)
print(f'Saved → {out}')

---
## Metric 2 — Compute Time per Image

Reports total elapsed time divided by the number of images processed, in **seconds / image**.
Time is configuration-wise (AXps and CXps are enumerated together; no separate breakdown is available).
For XpSatEnum, the reported time covers only the saturation step (on top of XpEnum).

> `*` = partial run (timeout reached before all images were processed)   `-` = not available

In [ ]:
from analysis.compute_time import compute_time_per_image

df_m2 = compute_time_per_image(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)

caption = f'Compute time (s/img) — behavior {behavior} ({model}, {dataset}, N={vocab_size})'
df_m2.style.set_caption(caption)

In [ ]:
out = RESULTS_DIR / f'metric2_{behavior}.csv'
df_m2.to_csv(out)
print(f'Saved → {out}')

---
## Metric 3 — Individual Coverage

**Individual coverage** of explanation *E* = (# images where *E* appears) / (# images processed).
A value of 1 means *E* accounts for every image in the behavior.

**Table (3a):** top-10 explanations by coverage for AXps and CXps separately.

**Plot (3b):** coverage decay curve — rank on the x-axis (1–50), coverage on the y-axis (0–1).
Color encodes the erasure algorithm; line style encodes the enumeration algorithm.

> `*` = partial run   `-` = not available

In [ ]:
from analysis.individual_coverage import compute_individual_coverage_table, plot_coverage_curves

df_axp, df_cxp = compute_individual_coverage_table(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)

display(df_axp.style.set_caption(
    f'Top-10 AXps Individual Coverage — behavior {behavior} ({model}, {dataset}, N={vocab_size})'
))
df_cxp.style.set_caption(
    f'Top-10 CXps Individual Coverage — behavior {behavior} ({model}, {dataset}, N={vocab_size})'
)

In [ ]:
fig = plot_coverage_curves(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)
fig

In [ ]:
df_axp.to_csv(RESULTS_DIR / f'metric3_axp_{behavior}.csv')
df_cxp.to_csv(RESULTS_DIR / f'metric3_cxp_{behavior}.csv')
fig.savefig(RESULTS_DIR / f'metric3_coverage_curves_{behavior}.png', dpi=150, bbox_inches='tight')
print(f'Saved tables and plot to {RESULTS_DIR}/')

---
## Metric 4 — Maximum Coverage at K

**Maximum Coverage at K** selects at most K explanations (greedily, by marginal gain)
to maximize the number of images covered. Normalized by total images (1.0 = full behavior covered).

The greedy strategy: at each step pick the explanation that covers the most previously uncovered
images. This gives a (1 − 1/e)-approximation to the NP-hard optimum.

Color encodes the erasure algorithm; line style encodes the enumeration algorithm.

In [ ]:
from analysis.max_coverage import plot_max_coverage

fig_m4 = plot_max_coverage(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)
fig_m4

In [ ]:
fig_m4.savefig(RESULTS_DIR / f'metric4_max_coverage_{behavior}.png', dpi=150, bbox_inches='tight')
print(f'Saved → {RESULTS_DIR}/metric4_max_coverage_{behavior}.png')

---
## Metric 5 — Explanation Size vs. Individual Coverage

Violin plots comparing individual coverage distributions for explanations of size
**L=1**, **L=2**, and **L=3+** (number of concepts, regardless of sign).

X-axis has three levels: overall concept label → available (eraser / algorithm / xp_type)
configurations → size group. Violin color = erasure algorithm. Strip plots are used
when a size group has fewer than 5 unique explanations.

In [ ]:
from analysis.size_vs_coverage import plot_size_vs_coverage

fig_m5 = plot_size_vs_coverage(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)
fig_m5

In [ ]:
fig_m5.savefig(RESULTS_DIR / f'metric5_size_vs_coverage_{behavior}.png', dpi=150, bbox_inches='tight')
print(f'Saved → {RESULTS_DIR}/metric5_size_vs_coverage_{behavior}.png')

---
## Metric 6 — Generalizability at K

**Generalizability at K** measures how consistently the top-K explanations
appear across two random 50/50 splits of the behavior images:

    IoU(topK_half1, topK_half2) = |topK_h1 ∩ topK_h2| / |topK_h1 ∪ topK_h2|

A value of 1.0 means the same K explanations rank highest in both halves;
0.0 means the two halves agree on none of the top-K.

**XpSatEnum is excluded** from this metric (it builds on XpEnum; a dedicated
script in `src/generalizability_xpsatenum.py` handles it separately).
Configurations with fewer than 20 image blocks are skipped.

> Color = erasure algorithm (orange = Ortho, purple = SpLiCE, green = LEACE)

In [ ]:
from analysis.generalizability import plot_generalizability

fig_m6 = plot_generalizability(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)
fig_m6

In [ ]:
fig_m6.savefig(RESULTS_DIR / f'metric6_generalizability_{behavior}.png', dpi=150, bbox_inches='tight')
print(f'Saved → {RESULTS_DIR}/metric6_generalizability_{behavior}.png')

---
## Metric 7 — Validity Ratio

**Validity ratio** measures the *plausibility* of an explanation relative to
LLM-graded concept relevance for the behavior.

For a signed explanation with positive concepts `P` and negative concepts `N`:

    validity_ratio = (|{p ∈ P : p is RELEVANT}| + |{n ∈ N : n is IRRELEVANT}|)
                     ──────────────────────────────────────────────────────────────
                                           |P| + |N|

Three bins (y-axis = fraction of explanation *appearances*):
- **Fully Plausible** (green, ≥ 0.99) — all signed concepts agree with LLM relevance
- **Partially Plausible** (teal, 0.01 – 0.99) — mixed agreement
- **Fully Implausible** (red, ≤ 0.01) — all signed concepts disagree

Explanations with higher individual coverage contribute proportionally more to the fractions.

Within each (Eraser × Algorithm) group, AXps (solid bars) and CXps (hatched bars)
are shown side by side for each plausibility category.

> **Prerequisite:** generate concept relevance before running this cell:  
> `python src/classify_concepts.py --dataset rival10 --behavior B26`  
> Requires `ANTHROPIC_API_KEY` in the environment.

In [ ]:
from analysis.validity_ratio import plot_validity_ratio

fig_m7 = plot_validity_ratio(
    model=model, dataset=dataset, vocab_size=vocab_size, behavior=behavior,
    intermediate_results_dir=INTERMEDIATE_RESULTS,
)
fig_m7

In [ ]:
fig_m7.savefig(RESULTS_DIR / f'metric7_validity_ratio_{behavior}.png', dpi=150, bbox_inches='tight')
print(f'Saved → {RESULTS_DIR}/metric7_validity_ratio_{behavior}.png')